In [ ]:
# Install the package in development mode if needed
# !pip install -e '.[mcp]'

import asyncio
import os
import sys
import logging
from pathlib import Path
import json

# Import the MCP components
from napistu.mcp.server import create_server, initialize_components
from napistu.mcp import documentation, codebase, tutorials, execution
from napistu.mcp.config import local_server_config
from napistu.mcp.semantic_search import SemanticSearch

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("napistu")

# Helper function to run async code in Jupyter
async def run_async(coro):
    return await coro

# Create a dummy session context for execution components
session_context = {}
object_registry = {}

## Setting up a local test server

In [ ]:
from napistu.mcp.profiles import get_profile
# define the types of assets to load
profile = get_profile("full")

In [ ]:
# register the relevant components
mcp = create_server(profile, local_server_config())
# initialize the relevant components
live_server = await run_async(initialize_components(profile))

In [ ]:
import shutil

## Component-level testing

In [ ]:
# shared semantic search instance
semantic_search = SemanticSearch()

In [ ]:
# documentation component

doc_component = documentation.get_component()
print(f"Component created: {type(doc_component)}")
print(f"State initialized: {doc_component.state.initialized}")

async def test_initialization():
    """Test component initialization."""
    print("Starting initialization...")
    success = await doc_component.safe_initialize(semantic_search)
    print(f"Initialization success: {success}")
    
    # Check state
    state = doc_component.get_state()
    print(f"Component healthy: {state.is_healthy()}")
    print(f"Health details: {state.get_health_details()}")
    
    return success

# Run initialization
success = await test_initialization()

if success:
    # Test that issues/PRs are now properly indexed
    QUERY = "how do i use sbml_dfs"
    results = semantic_search.search(QUERY, "documentation", n_results=5)
    print(f"\nTest results for '{QUERY}':")
    for i, result in enumerate(results, 1):
        print(f"{i}. {result['source']}. Score: {result['similarity_score']}")
        print(f"   {result['content'][:100]}...")

In [ ]:
# tutorials

tutorials_component = tutorials.get_component()
print(f"Component created: {type(tutorials_component)}")
print(f"State initialized: {tutorials_component.state.initialized}")

async def test_initialization():
    """Test component initialization."""
    print("Starting initialization...")
    success = await tutorials_component.safe_initialize(semantic_search)
    print(f"Initialization success: {success}")
    
    # Check state
    state = tutorials_component.get_state()
    print(f"Component healthy: {state.is_healthy()}")
    print(f"Health details: {state.get_health_details()}")
    
    return success

# Run initialization
success = await test_initialization()

if success:
    # Test that issues/PRs are now properly indexed
    QUERY = "how do i use sbml_dfs"
    results = semantic_search.search(QUERY, "tutorials", n_results=5)
    print(f"\nTest results for '{QUERY}':")
    for i, result in enumerate(results, 1):
        print(f"{i}. {result['source']}. Score: {result['similarity_score']}")
        print(f"   {result['content'][:100]}...")

In [ ]:
# codebase

codebase_component = codebase.get_component()
print(f"Component created: {type(codebase_component)}")
print(f"State initialized: {codebase_component.state.initialized}")

async def test_initialization():
    """Test component initialization."""
    print("Starting initialization...")
    success = await codebase_component.safe_initialize(semantic_search)
    print(f"Initialization success: {success}")
    
    # Check state
    state = codebase_component.get_state()
    print(f"Component healthy: {state.is_healthy()}")
    print(f"Health details: {state.get_health_details()}")
    
    return success

# Run initialization
success = await test_initialization()

if success:
    # Test that issues/PRs are now properly indexed
    QUERY = "how do i use sbml_dfs"
    results = semantic_search.search(QUERY, "codebase", n_results=5)
    print(f"\nTest results for '{QUERY}':")
    for i, result in enumerate(results, 1):
        print(f"{i}. {result['source']}. Score: {result['similarity_score']}")
        print(f"   {result['content'][:100]}...")

In [ ]:
semantic_search.search(QUERY, "documentation", n_results=5)
semantic_search.search(QUERY, "tutorials", n_results=5)

## ETLing specific content 

In [ ]:
from napistu.mcp import documentation_utils
from napistu.mcp.codebase_utils import read_read_the_docs

from napistu.mcp.constants import READMES
from napistu.mcp.constants import NAPISTU_PY_READTHEDOCS_API

In [ ]:
# readmes
readme = await run_async(documentation_utils.load_readme_content(READMES["napistu"]))
readme


In [ ]:
# read-the-docs module and function defs
rtd_docs = await run_async(read_read_the_docs(package_toc_url = NAPISTU_PY_READTHEDOCS_API))
rtd_docs


In [ ]:
# github wiki

from napistu.constants import PACKAGE_DEFS

wiki_pages = await run_async(documentation_utils.list_wiki_pages(repo = PACKAGE_DEFS.GITHUB_PROJECT_REPO))

In [ ]:
# github issues and PRs

GITHUB_ISSUES_INDEXED = "all"
GITHUB_PRS_INDEXED = "all"

# load issues (already includes the body)
issue_list = await run_async(documentation_utils.list_issues(PACKAGE_DEFS.GITHUB_PROJECT_REPO, state = GITHUB_ISSUES_INDEXED))

# load PRs
prs_list = await run_async(documentation_utils.list_pull_requests(PACKAGE_DEFS.GITHUB_PROJECT_REPO, state = GITHUB_PRS_INDEXED))

# there is probably no reason to use this since this info is captured in the lists
# prs_list = await run_async(get_issue(PACKAGE_DEFS.GITHUB_PROJECT_REPO, number = 11))


In [ ]:
# tutorials

from napistu.mcp.constants import TUTORIAL_URLS
from napistu.mcp.tutorials_utils import get_tutorial_markdown

In [ ]:
tutorial_markdown_dict = dict()
for k, v in TUTORIAL_URLS.items():
    tutorial_markdown_dict[k] = await run_async(get_tutorial_markdown(k))

tutorial_markdown_dict

# Client operations

Run an MCP cleint talking to either a local or remote server

In [1]:
from napistu.mcp.client import check_server_health, list_server_resources, read_server_resource, call_server_tool
from napistu.mcp.config import local_client_config, production_client_config

# Health check (production server)
#config = production_client_config()
#health = await check_server_health(config)
#print(f"Server status: {health['status']}")

# List resources
config = local_client_config()
resources = await list_server_resources(config)
for resource in resources:
    print(f"Available: {resource.uri}")

# Read specific resource  
config = local_client_config()
content = await read_server_resource("napistu://health", config)

Available: napistu://documentation/summary
Available: napistu://codebase/summary
Available: napistu://tutorials/index
Available: napistu://health


In [ ]:
# Search using the unified component interface
from napistu.mcp.client import search_component

result = await search_component("codebase", "SBML_dfs validation methods", config=config)

# Call any MCP tool directly  
from napistu.mcp.client import call_server_tool

result = await call_server_tool(
    "search_documentation",
    {"query": "what does Napistu mean?", "search_type": "semantic"},
    config
)

In [6]:
result = await call_server_tool(
    "get_class_documentation",
    {"class_name": "SBML_dfs"},
    config
)
result

{'name': 'SBML_dfs',
 'signature': 'classnapistu.sbml_dfs_core.SBML_dfs(sbml_model:sbml.SBML|MutableMapping[str,pd.DataFrame|dict[str,pd.DataFrame]],model_source:source.Source,validate:bool=True,resolve:bool=True)',
 'id': 'napistu.sbml_dfs_core.SBML_dfs',
 'doc': "Bases: object System Biology Markup Language Model Data Frames. A class representing a SBML model as a collection of pandas DataFrames.\nThis class provides methods for manipulating and analyzing biological pathway models\nwith support for species, reactions, compartments, and their relationships. compartments \uf0c1 Sub-cellular compartments in the model, indexed by compartment ID (c_id) Type : pd.DataFrame species \uf0c1 Molecular species in the model, indexed by species ID (s_id) Type : pd.DataFrame species_data \uf0c1 Additional data for species. Each DataFrame is indexed by species_id (s_id) Type : Dict[str, pd.DataFrame] reactions \uf0c1 Reactions in the model, indexed by reaction ID (r_id) Type : pd.DataFrame reaction

In [4]:
result = await call_server_tool(
    "get_function_documentation",
    {"function_name": "unnest_sources"},
    config
)
result

{'name': 'unnest_sources',
 'signature': 'napistu.source.unnest_sources(source_table:DataFrame,verbose:bool=False)→DataFrame',
 'id': 'napistu.source.unnest_sources',
 'doc': 'Unnest Sources Take a pd.DataFrame containing an array of Sources and\nreturn one-row per source. Parameters : source_table ( pd.DataFrame ) – a table containing an array of Sources verbose ( bool ) – print progress Returns : pd.Dataframe containing the index of source_table but expanded to include one row per source',
 'stripped_name': 'unnest_sources',
 'full_name': 'napistu.source.unnest_sources'}